#  Qualité et compréhension des données

##  Contexte du projet

###  Présentation du projet

Ce projet porte sur la maintenance prédictive d'un compresseur industriel à partir du jeu de données MetroPT-3.

L'objectif est d'analyser les mesures issues des capteurs, de comprendre le comportement opérationnel du compresseur et de préparer des données fiables pour le développement de modèles de Machine Learning.

###  Source des données

- **Jeu de données :** MetroPT-3
- **Source :** UCI Machine Learning Repository
- **Fichier :** `MetroPT3(AirCompressor).csv`
- **Système étudié :** système de production d'air comprimé du métro de Porto.

###  Objectifs du notebook

Ce notebook est exclusivement consacré à la compréhension du jeu de données et à l'évaluation de sa qualité structurelle.

Les principaux objectifs sont :

- Identifier la structure et les dimensions du jeu de données.
- Comprendre les variables disponibles et leurs types.
- Détecter les valeurs manquantes et les observations dupliquées.
- Identifier les variables constantes et les valeurs infinies.
- Vérifier l'intégrité des données temporelles.
- Établir un bilan de la qualité structurelle des données.

### Périmètre du notebook

Aucune ingénierie de variables, construction de la cible, transformation des données ou modélisation ne sera réalisée dans ce notebook.

Ces opérations seront traitées dans des notebooks dédiés.


In [1]:
# Manipulation des données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Gestion des chemins
from pathlib import Path

# Configuration de l'affichage
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

print("Environnement initialisé avec succès.")


Environnement initialisé avec succès.



##  Chargement et présentation des données

###  Objectif

Charger le jeu de données MetroPT-3 original et examiner sa structure générale avant de réaliser les contrôles détaillés de qualité.

###  Première exploration

Les éléments suivants seront examinés :

- Dimensions du jeu de données.
- Premières et dernières observations.
- Noms des colonnes.
- Types de données.
- Occupation mémoire.

In [2]:
import yaml

CHEMIN_CONFIG = Path("../config.yaml")

if not CHEMIN_CONFIG.exists():
    raise FileNotFoundError(
        f"Fichier de configuration introuvable : "
        f"{CHEMIN_CONFIG.resolve()}"
    )

with open(CHEMIN_CONFIG, "r", encoding="utf-8") as fichier:
    config = yaml.safe_load(fichier)

print("Configuration chargée avec succès.")


Configuration chargée avec succès.


In [18]:
CHEMIN_DONNEES = (
    Path("../")
    / config["data"]["raw_path"]
    / config["data"]["dataset_name"]
)

if not CHEMIN_DONNEES.exists():
    raise FileNotFoundError(
        f"Dataset introuvable : {CHEMIN_DONNEES.resolve()}"
    )

df = pd.read_csv(CHEMIN_DONNEES)

print("Dataset chargé avec succès.")
print(f"Fichier utilisé : {CHEMIN_DONNEES}")
print(f"Dataset dimension  : {df.shape}")
print(f"Dimension du Dataset : {df.info()}")
print(df.head(5))
print(df.tail(5))


Dataset chargé avec succès.
Fichier utilisé : ..\data\raw\MetroPT3(AirCompressor).csv
Dataset dimension  : (1516948, 17)
<class 'pandas.DataFrame'>
RangeIndex: 1516948 entries, 0 to 1516947
Data columns (total 17 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   Unnamed: 0       1516948 non-null  int64  
 1   timestamp        1516948 non-null  str    
 2   TP2              1516948 non-null  float64
 3   TP3              1516948 non-null  float64
 4   H1               1516948 non-null  float64
 5   DV_pressure      1516948 non-null  float64
 6   Reservoirs       1516948 non-null  float64
 7   Oil_temperature  1516948 non-null  float64
 8   Motor_current    1516948 non-null  float64
 9   COMP             1516948 non-null  float64
 10  DV_eletric       1516948 non-null  float64
 11  Towers           1516948 non-null  float64
 12  MPG              1516948 non-null  float64
 13  LPS              1516948 non-null  float64
 14  Pres


###  Résultats et interprétation

L'inspection du dataset permet de constater une dimension de (1516948, 17).



##  Analyse des types de données

### Objectif

Cette étape consiste à examiner les types de données de chaque colonne afin de vérifier leur cohérence avec la nature des variables.

Les principaux types recherchés sont :

- `int64` : nombres entiers.
- `float64` : nombres décimaux.
- `object` : chaînes de caractères ou données mixtes.
- `datetime64` : données temporelles.

Une attention particulière sera portée à la colonne temporelle, qui devra être interprétable comme une date et une heure.


In [4]:
types_donnees = pd.DataFrame({
    "Variable": df.columns,
    "Type": df.dtypes.astype(str).values,
    "Valeurs uniques": df.nunique(dropna=False).values,
    "Valeurs manquantes": df.isna().sum().values
})

display(types_donnees)

print("\nRépartition des types de données :")
display(
    df.dtypes.astype(str)
    .value_counts()
    .rename_axis("Type")
    .to_frame("Nombre de colonnes")
)


,Variable,Type,Valeurs uniques,Valeurs manquantes
0,Unnamed: 0,int64,1516948,0
1,timestamp,str,1516948,0
2,TP2,float64,5257,0
3,TP3,float64,3683,0
4,H1,float64,2665,0
5,DV_pressure,float64,2257,0
6,Reservoirs,float64,3682,0
7,Oil_temperature,float64,2462,0
8,Motor_current,float64,1809,0
9,COMP,float64,2,0



Répartition des types de données :


,Nombre de colonnes
Type,
float64,15
int64,1
str,1


In [5]:
COLONNE_TEMPORELLE = config["data"]["timestamp_column"]

if COLONNE_TEMPORELLE not in df.columns:
    raise KeyError(
        f"Colonne temporelle absente : {COLONNE_TEMPORELLE}"
    )

print("Type actuel :",
      df[COLONNE_TEMPORELLE].dtype)

print("Exemples de valeurs :")
display(df[COLONNE_TEMPORELLE].head())


Type actuel : str
Exemples de valeurs :


0    2020-02-01 00:00:00
1    2020-02-01 00:00:10
2    2020-02-01 00:00:19
3    2020-02-01 00:00:29
4    2020-02-01 00:00:39
Name: timestamp, dtype: str


##  Analyse des valeurs manquantes

###  Objectif

Identifier les valeurs manquantes dans chaque variable afin d'évaluer la complétude du jeu de données.

Cette analyse permettra de déterminer :
- Le nombre de valeurs manquantes par variable.
- Le pourcentage de valeurs manquantes.
- Les variables potentiellement concernées par un problème de complétude.

In [6]:
analyse_manquants = pd.DataFrame({
    "Nombre_manquants": df.isna().sum(),
    "Pourcentage_manquants": (
        df.isna().mean() * 100
    )
})

analyse_manquants = analyse_manquants.sort_values(
    by="Pourcentage_manquants",
    ascending=False
)

display(analyse_manquants)

total_manquants = df.isna().sum().sum()

print(f"Nombre total de valeurs manquantes : {total_manquants:,}")
print(
    f"Pourcentage global : "
    f"{(total_manquants / df.size) * 100:.4f} %"
)

,Nombre_manquants,Pourcentage_manquants
Unnamed: 0,0,0.0
timestamp,0,0.0
TP2,0,0.0
TP3,0,0.0
H1,0,0.0
DV_pressure,0,0.0
Reservoirs,0,0.0
Oil_temperature,0,0.0
Motor_current,0,0.0
COMP,0,0.0


Nombre total de valeurs manquantes : 0
Pourcentage global : 0.0000 %



###  Résultats et interprétation

L'analyse des valeurs manquantes ne révèle aucune donnée manquante dans le jeu de données original.

Le taux global de valeurs manquantes est de 0 %.

Aucune opération d'imputation ou de suppression liée aux valeurs manquantes n'est donc nécessaire à ce stade.

**Conclusion :** le jeu de données présente une complétude structurelle satisfaisante selon ce contrôle.


In [7]:
nb_doublons_complets = df.duplicated().sum()

pourcentage_doublons = (
    nb_doublons_complets / len(df)
) * 100

print(
    f"Nombre de lignes dupliquées : "
    f"{nb_doublons_complets:,}"
)

print(
    f"Pourcentage de doublons : "
    f"{pourcentage_doublons:.4f} %"
)


Nombre de lignes dupliquées : 0
Pourcentage de doublons : 0.0000 %


In [8]:
COLONNE_TEMPORELLE = config["data"]["timestamp_column"]

nb_timestamps_dupliques = (
    df[COLONNE_TEMPORELLE].duplicated().sum()
)

print(
    f"Nombre de timestamps dupliqués : "
    f"{nb_timestamps_dupliques:,}"
)

# Afficher les observations concernées
timestamps_dupliques = df[
    df[COLONNE_TEMPORELLE].duplicated(keep=False)
].sort_values(COLONNE_TEMPORELLE)

display(timestamps_dupliques.head(10))


Nombre de timestamps dupliqués : 0


,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses



###  Résultats et interprétation

L'analyse ne révèle aucun doublon complet dans le jeu de données.

De même, aucune répétition de timestamp n'a été détectée.

Chaque observation possède donc une combinaison de valeurs unique et chaque timestamp apparaît une seule fois.

**Conclusion :** aucune anomalie liée aux doublons n'a été identifiée. Aucune suppression de données n'est nécessaire à ce stade.


In [9]:
analyse_variabilite = pd.DataFrame({
    "Nombre_valeurs_uniques": df.nunique(dropna=False),
    "Valeur_dominante": df.apply(
        lambda colonne: colonne.value_counts(dropna=False).index[0]
    ),
    "Frequence_dominante_pct": df.apply(
        lambda colonne: (
            colonne.value_counts(dropna=False).iloc[0]
            / len(colonne)
        ) * 100
    )
})

variables_constantes = analyse_variabilite[
    analyse_variabilite["Nombre_valeurs_uniques"] == 1
]

display(analyse_variabilite.sort_values(
    "Frequence_dominante_pct",
    ascending=False
))

print(
    f"\nNombre de variables constantes : "
    f"{len(variables_constantes)}"
)

display(variables_constantes)

,Nombre_valeurs_uniques,Valeur_dominante,Frequence_dominante_pct
LPS,2,0.0,99.657998
Pressure_switch,2,1.0,99.143675
Caudal_impulses,2,1.0,93.710661
Towers,2,1.0,91.984827
Oil_level,2,1.0,90.415558
DV_eletric,2,0.0,83.938935
COMP,2,1.0,83.695684
MPG,2,1.0,83.266401
TP2,5257,-0.012,29.369431
DV_pressure,2257,-0.02,21.241400



Nombre de variables constantes : 0


,Nombre_valeurs_uniques,Valeur_dominante,Frequence_dominante_pct


In [10]:
SEUIL_VARIABILITE = 99.9  # Seuil exploratoire en %

variables_quasi_constantes = analyse_variabilite[
    (analyse_variabilite["Frequence_dominante_pct"] >= SEUIL_VARIABILITE)
    & (analyse_variabilite["Nombre_valeurs_uniques"] > 1)
]

print(
    f"Seuil exploratoire : {SEUIL_VARIABILITE} %"
)

print(
    f"Nombre de variables quasi constantes : "
    f"{len(variables_quasi_constantes)}"
)

display(variables_quasi_constantes)


Seuil exploratoire : 99.9 %
Nombre de variables quasi constantes : 0


,Nombre_valeurs_uniques,Valeur_dominante,Frequence_dominante_pct



###  Résultats et interprétation

L'analyse de la variabilité ne révèle aucune variable constante dans le jeu de données.

Aucune variable ne présente une fréquence dominante supérieure ou égale à 99,9 %.

Toutes les variables présentent donc une certaine variabilité selon les critères retenus.

**Conclusion :** aucune variable n'est à exclure sur la seule base de son absence de variabilité ou de son caractère quasi constant.


In [11]:
colonnes_numeriques = df.select_dtypes(
    include=["number"]
).columns

masque_infini = np.isinf(df[colonnes_numeriques])

analyse_infinis = pd.DataFrame({
    "Valeurs_positives_inf": np.isposinf(
        df[colonnes_numeriques]
    ).sum(),
    "Valeurs_negatives_inf": np.isneginf(
        df[colonnes_numeriques]
    ).sum()
})

analyse_infinis["Total_infinis"] = (
    analyse_infinis["Valeurs_positives_inf"]
    + analyse_infinis["Valeurs_negatives_inf"]
)

display(analyse_infinis)

total_infinis = analyse_infinis["Total_infinis"].sum()

print(f"\nNombre total de valeurs infinies : {total_infinis:,}")


,Valeurs_positives_inf,Valeurs_negatives_inf,Total_infinis
Unnamed: 0,0,0,0
TP2,0,0,0
TP3,0,0,0
H1,0,0,0
DV_pressure,0,0,0
Reservoirs,0,0,0
Oil_temperature,0,0,0
Motor_current,0,0,0
COMP,0,0,0
DV_eletric,0,0,0



Nombre total de valeurs infinies : 0



### Résultats et interprétation

La recherche des valeurs infinies ne révèle aucune valeur positive ou négative infinie dans les colonnes numériques.

Aucune anomalie de ce type n'a donc été détectée.

**Conclusion :** aucune correction liée aux valeurs infinies n'est nécessaire à ce stade.



##  Analyse de l'intégrité temporelle

###  Objectif

Cette étape vise à vérifier la cohérence temporelle des observations.

La colonne temporelle, initialement chargée sous forme de chaîne de caractères, doit être convertie en format datetime afin de permettre les analyses chronologiques.

Les contrôles porteront sur :
- La validité des dates.
- L'ordre chronologique des observations.
- Les intervalles entre deux mesures successives.
- Les éventuelles interruptions dans l'acquisition des données.


In [12]:
# Création d'une copie de travail
df_temporel = df.copy()

COLONNE_TEMPORELLE = config["data"]["timestamp_column"]

# Conversion en datetime
df_temporel[COLONNE_TEMPORELLE] = pd.to_datetime(
    df_temporel[COLONNE_TEMPORELLE],
    errors="coerce"
)

# Détection des dates invalides
nb_dates_invalides = (
    df_temporel[COLONNE_TEMPORELLE].isna().sum()
)

print(f"Nombre de dates invalides : {nb_dates_invalides:,}")

# Période couverte
print(
    "\nDate minimale :",
    df_temporel[COLONNE_TEMPORELLE].min()
)

print(
    "Date maximale :",
    df_temporel[COLONNE_TEMPORELLE].max()
)


Nombre de dates invalides : 0

Date minimale : 2020-02-01 00:00:00
Date maximale : 2020-09-01 03:59:50


In [13]:
timestamps = df_temporel[COLONNE_TEMPORELLE]

ordre_chronologique = timestamps.is_monotonic_increasing

print(
    "Les observations sont-elles chronologiques ?",
    ordre_chronologique
)

# Nombre de ruptures dans l'ordre chronologique
nb_ruptures = timestamps.diff().lt(
    pd.Timedelta(0)
).sum()

print(f"Nombre de ruptures chronologiques : {nb_ruptures:,}")


Les observations sont-elles chronologiques ? True
Nombre de ruptures chronologiques : 0



###  Résultats et interprétation

La conversion de la colonne temporelle ne révèle aucune date invalide.

La période couverte par le jeu de données s'étend du 1er février 2020 à 00:00:00 au 1er septembre 2020 à 03:59:50.

Les observations sont déjà classées dans l'ordre chronologique. Aucune rupture dans cet ordre n'a été détectée.

**Conclusion :** la colonne temporelle présente une bonne cohérence structurelle. Les données peuvent être utilisées pour les analyses chronologiques, sous réserve de vérifier la régularité des intervalles d'acquisition.


In [14]:
# Calcul des écarts entre observations successives
intervalles = (
    df_temporel[COLONNE_TEMPORELLE]
    .diff()
    .dt.total_seconds()
    .dropna()
)

# Statistiques descriptives
statistiques_intervalles = intervalles.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

display(statistiques_intervalles.to_frame("Intervalle_secondes"))

# Fréquence des intervalles les plus courants
print("\nIntervalles les plus fréquents :")
display(
    intervalles.value_counts()
    .head(10)
    .rename_axis("Intervalle_secondes")
    .to_frame("Nombre_occurrences")
)


,Intervalle_secondes
count,1.516947e+06
mean,1.214122e+01
std,3.141073e+02
min,8.000000e+00
25%,1.000000e+01
50%,1.000000e+01
75%,1.000000e+01
90%,1.000000e+01
95%,1.000000e+01
99%,1.200000e+01



Intervalles les plus fréquents :


,Nombre_occurrences
Intervalle_secondes,
10.0,1337521
9.0,128277
12.0,38321
13.0,7988
11.0,4471
21.0,10
19.0,5
22.0,4
20.0,3


In [15]:
# Seuils exploratoires en secondes
seuils = {
    "Supérieur à 10 secondes": 10,
    "Supérieur à 1 minute": 60,
    "Supérieur à 5 minutes": 300,
    "Supérieur à 1 heure": 3600
}

resultats_interruptions = []

for nom, seuil in seuils.items():

    interruptions = intervalles[intervalles > seuil]

    resultats_interruptions.append({
        "Critère": nom,
        "Nombre": len(interruptions),
        "Pourcentage": (
            len(interruptions) / len(intervalles)
        ) * 100,
        "Durée_maximale_secondes": (
            interruptions.max() if len(interruptions) > 0 else 0
        )
    })

df_interruptions = pd.DataFrame(resultats_interruptions)

display(df_interruptions)


,Critère,Nombre,Pourcentage,Durée_maximale_secondes
0,Supérieur à 10 secondes,51147,3.371706,172918.0
1,Supérieur à 1 minute,331,0.021820,172918.0
2,Supérieur à 5 minutes,275,0.018129,172918.0
3,Supérieur à 1 heure,160,0.010548,172918.0


In [16]:
# Reconstituer les intervalles avec leurs timestamps
analyse_intervalles = pd.DataFrame({
    "timestamp_debut": df_temporel[COLONNE_TEMPORELLE].shift(1),
    "timestamp_fin": df_temporel[COLONNE_TEMPORELLE],
    "intervalle_secondes": intervalles
}).dropna()

# Sélectionner les interruptions supérieures à 1 heure
interruptions_importantes = (
    analyse_intervalles[
        analyse_intervalles["intervalle_secondes"] > 3600
    ]
    .sort_values(
        by="intervalle_secondes",
        ascending=False
    )
)

display(interruptions_importantes.head(20))


,timestamp_debut,timestamp_fin,intervalle_secondes
618538,2020-04-25 01:10:51,2020-04-27 01:12:49,172918.0
1055664,2020-06-27 10:53:07,2020-06-28 23:07:43,130476.0
214850,2020-02-28 23:57:08,2020-03-01 04:00:09,100981.0
1318888,2020-08-04 07:42:28,2020-08-05 08:23:01,88833.0
805415,2020-05-24 00:39:23,2020-05-25 01:14:14,88491.0
1126187,2020-07-07 15:24:51,2020-07-08 15:20:51,86160.0
1454438,2020-08-22 19:11:44,2020-08-23 18:51:01,85157.0
708881,2020-05-10 00:31:17,2020-05-10 22:48:58,80261.0
908125,2020-06-07 14:19:39,2020-06-08 11:48:04,77305.0
450114,2020-04-01 13:15:40,2020-04-02 09:59:17,74617.0


In [17]:
# Sélection des interruptions supérieures à une heure
interruptions_par_mois = interruptions_importantes.copy()

interruptions_par_mois["mois"] = (
    interruptions_par_mois["timestamp_fin"]
    .dt.to_period("M")
)

# Nombre d'interruptions par mois
repartition_mensuelle = (
    interruptions_par_mois
    .groupby("mois")
    .agg(
        nombre_interruptions=("intervalle_secondes", "count"),
        duree_totale_heures=("intervalle_secondes",
                             lambda x: x.sum() / 3600),
        duree_maximale_heures=("intervalle_secondes",
                               lambda x: x.max() / 3600)
    )
    .reset_index()
)

display(repartition_mensuelle)


,mois,nombre_interruptions,duree_totale_heures,duree_maximale_heures
0,2020-02,20,77.615278,7.976111
1,2020-03,21,130.115833,28.050278
2,2020-04,26,158.210833,48.032778
3,2020-05,28,144.399167,24.580833
4,2020-06,17,104.270278,36.243333
5,2020-07,24,120.808056,23.933333
6,2020-08,22,126.802778,24.675833
7,2020-09,2,6.290833,4.373056



##  Dictionnaire des variables — MetroPT-3

Le jeu de données MetroPT-3 contient 15 variables issues des capteurs d'un système industriel de production d'air comprimé. Ces variables permettent de suivre les paramètres physiques et les états de fonctionnement du compresseur.

### Variables numériques analogiques

Ces variables représentent des mesures physiques continues.

| Variable | Désignation | Unité |
|---|---|---|
| `TP2` | Pression au niveau du compresseur | bar |
| `TP3` | Pression au niveau du panneau pneumatique | bar |
| `H1` | Pression liée à la décharge du filtre séparateur cyclonique | bar |
| `DV_pressure` | Chute de pression au niveau des sécheurs d'air | bar |
| `Reservoirs` | Pression en aval des réservoirs d'air | bar |
| `Oil_temperature` | Température de l'huile du compresseur | °C |
| `Motor_current` | Courant électrique d'une phase du moteur triphasé | A |

###  Variables numériques digitales

Ces variables représentent des états ou des signaux de commande et de détection. Elles sont généralement codées sous forme binaire (0 et 1).

| Variable | Désignation | Unité |
|---|---|---|
| `COMP` | Signal électrique de la vanne d'admission d'air | Sans unité |
| `DV_eletric` | Signal électrique de commande de la vanne de sortie du compresseur | Sans unité |
| `Towers` | État des tours du système de séchage d'air | Sans unité |
| `MPG` | Signal de déclenchement du compresseur selon la pression | Sans unité |
| `LPS` | Signal de détection d'une pression inférieure à 7 bar | Sans unité |
| `Pressure_switch` | Signal de détection d'une décharge au niveau de la vanne de pilotage | Sans unité |
| `Oil_level` | Signal indiquant un niveau d'huile inférieur au seuil attendu | Sans unité |
| `Caudal_impulses` | Signal associé aux impulsions du débitmètre d'air | Sans unité |

### Variables temporelles et de traçabilité

Ces variables permettent d'identifier et de suivre les observations.

| Variable | Désignation | Nature |
|---|---|---|
| `timestamp` | Date et heure d'acquisition de la mesure | Temporelle |
| `Unnamed: 0` | Index d'origine présent dans le fichier CSV | Identifiant technique |

###  Variables construites pour le projet

Ces variables ont été ajoutées lors de la préparation du jeu de données.

| Variable | Désignation | Nature |
|---|---|---|
| `target_event` | Indique si l'observation appartient à l'une des quatre périodes de fuite documentées | Binaire (0/1) |
| `event_id` | Identifiant de l'événement associé à une observation positive | Identifiant (1 à 4) |

###  Synthèse

| Catégorie | Nombre de variables |
|---|---:|
| Numériques analogiques | 7 |
| Numériques digitales | 8 |
| Temporelles et traçabilité | 2 |
| Variables construites | 2 |
| **Total** | **19** |

**Remarque :** les 15 premières variables correspondent aux capteurs originaux. Les 4 autres colonnes correspondent à la traçabilité et aux éléments construits pour le projet.

Les variables digitales sont stockées sous forme numérique, mais leur interprétation dépend de leur signification physique. `Caudal_impulses` doit notamment être distinguée d'une mesure directe de débit.



##  Bilan de la qualité des données

###  Synthèse des contrôles

L'analyse structurelle du jeu de données MetroPT-3 a permis d'obtenir les résultats suivants :

- **Valeurs manquantes :** aucune valeur manquante détectée.
- **Doublons complets :** aucun doublon détecté.
- **Doublons temporels :** aucun timestamp dupliqué.
- **Variables constantes :** aucune variable constante ou quasi constante selon le seuil exploratoire de 99,9 %.
- **Valeurs infinies :** aucune valeur infinie détectée.
- **Dates invalides :** aucune date invalide détectée.
- **Ordre chronologique :** les observations sont déjà classées chronologiquement.
- **Fréquence d'acquisition :** intervalle habituel de 10 secondes.
- **Interruptions :** plusieurs interruptions d'acquisition ont été identifiées, dont certaines dépassent 24 heures.

###  Conclusion générale

Le jeu de données présente une bonne qualité structurelle selon les contrôles réalisés.

Aucune anomalie liée aux valeurs manquantes, aux doublons, aux variables constantes ou aux valeurs infinies n'a été détectée.

Cependant, l'acquisition temporelle présente des interruptions ponctuelles importantes. Ces interruptions devront être prises en compte lors des analyses temporelles et de la définition de la stratégie de validation des modèles.

Aucune observation n'a été supprimée ou modifiée au cours de cette phase de diagnostic.

###  Décision

Le jeu de données peut être utilisé pour poursuivre l'analyse exploratoire et la préparation du problème de maintenance prédictive, en conservant une traçabilité des interruptions temporelles identifiées.
